In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-03-17_OPhys_LabMeetingIV\figures"

In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

In [ ]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check","volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [ ]:
test_asset = assets[3]
test_asset.session_id

In [ ]:
test_asset

### Plot trial-wise response

In [ ]:
single_data_path = glob.glob(os.path.join(test_asset.derived_dir,'**','glutamate_single_trial_df.npz'))[0]
glu_qc_path = glob.glob(os.path.join(test_asset.qc_dir,'**','**glutamate**'))[0]

In [ ]:
single_data_path

In [ ]:
glu_single = np.load(single_data_path,allow_pickle=True)['data'][0]

In [ ]:
with open(glu_qc_path, "r") as f:
    glu_qc = json.load(f)

In [ ]:
im_names = list(glu_mean['DMD1']['image_identity'].keys())
colors = ['#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
 '#e1bee7', '#d7ccc8', '#cfd8dc', '#b2ebf2']
colors = colors[::-1]

In [ ]:
dmd = 1
syn = 2

fig, ax = plt.subplots(figsize=(4, 4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

all_traces = []

for i, im in enumerate(im_names):
    # adjust this key if your npz uses a different name than 'traces'
    trial_traces = glu_single[f'DMD{dmd}']['image_identity'][im]

    trial_traces = np.asarray(trial_traces)

    # expected shape options:
    # (n_trials, n_synapses, n_time)
    # (n_synapses, n_trials, n_time)
    # detect and adapt
    if trial_traces.ndim != 3:
        raise ValueError(f"Unexpected shape for {im}: {trial_traces.shape}")

    if trial_traces.shape[1] > syn and trial_traces.shape[0] != syn:
        # assume (n_trials, n_synapses, n_time)
        syn_traces = trial_traces[:, syn, :]
    elif trial_traces.shape[0] > syn:
        # assume (n_synapses, n_trials, n_time)
        syn_traces = trial_traces[syn, :, :]
    else:
        raise ValueError(f"Could not infer synapse axis for {im}: {trial_traces.shape}")

    time = np.linspace(-0.25, 0.5, syn_traces.shape[-1])

    first_for_legend = True
    for tr in syn_traces[:3]:
        tr_smooth = pd.Series(tr).rolling(3, min_periods=1).mean().to_numpy()
        ax.plot(
            time,
            tr_smooth,
            lw=2,
            color=colors[i],
            alpha=0.5,
            )
        first_for_legend = False
        all_traces.append(tr)

# grand mean across all plotted trials
if len(all_traces) > 0:
    mean = np.mean([glu_mean[f'DMD{dmd}']['image_identity'][im_names[i]]['mean'][syn]for i,im in enumerate(im_names)],axis=0 )
    ax.plot(
        time,
        pd.Series(mean).rolling(1, min_periods=1).median().to_numpy(),
        color='k',
        lw=2.5,
        label='Mean'
    )

ax.axvline(0.0, color='k', dashes=[5, 3], lw=2)
ax.axvspan(0.0, 0.25, color='lightgray', alpha=0.4, zorder=0)

ax.set_xlabel('Time (s) from stimulus onset')
ax.set_ylabel('Single-trial ΔF')

for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2)

ax.set_title(f'Single-Trial Image Responses')
ax.legend(frameon=False, fontsize=10, ncol=2)
fig.tight_layout()

filen = '2026-03-30_single_trial_image_responses'
save_figure(fig, os.path.join(savepath, filen), formats=['.png', '.pdf'], dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# assumes:
# - glu_single already loaded
# - im_names already defined
# - colors already defined, same length as im_names
# - dmd, syn set

dmd = 1
syn = 2

pre_slice = slice(0, 50)
post_slice = slice(50, 100)

fig, ax = plt.subplots(figsize=(4, 4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for i, im in enumerate(im_names):
    trial_traces = glu_single[f'DMD{dmd}']['image_identity'][im]
    trial_traces = np.asarray(trial_traces)

    if trial_traces.ndim != 3:
        raise ValueError(f"Unexpected shape for {im}: {trial_traces.shape}")

    # handle either (n_trials, n_synapses, n_time) or (n_synapses, n_trials, n_time)
    if trial_traces.shape[1] > syn and trial_traces.shape[0] != syn:
        syn_traces = trial_traces[:, syn, :]
    elif trial_traces.shape[0] > syn:
        syn_traces = trial_traces[syn, :, :]
    else:
        raise ValueError(f"Could not infer synapse axis for {im}: {trial_traces.shape}")

    pre_auc = np.trapz(syn_traces[:, pre_slice], axis=1)
    post_auc = np.trapz(syn_traces[:, post_slice], axis=1)

    ax.scatter(
        pre_auc,
        post_auc,
        s=20,
        alpha=0.7,
        color=colors[i],
        label=im.split('/')[-1].split('\\')[-1],
        edgecolor='none'
    )

# unity line
all_vals = ax.get_xlim() + ax.get_ylim()
lo = min(all_vals)
hi = max(all_vals)
ax.plot([lo, hi], [lo, hi], color='k', ls='--', lw=1.5, alpha=0.8)

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)

ax.set_xlabel('Pre-stimulus AUC')
ax.set_ylabel('Post-stimulus AUC')
ax.set_title(f'Trial AUCs by image')
# ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1), loc='upper left')
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
    
fig.tight_layout()

filen = '2026-03-30_single_trial_auc_scatter'
save_figure(fig, os.path.join(savepath, filen), formats=['.png', '.pdf'], dpi=300)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pre_slice = slice(0, 50)
post_slice = slice(50, 100)

rows = []

for dmd in [1, 2]:
    dmd_key = f'DMD{dmd}'
    if dmd_key not in glu_single:
        continue

    im_names = list(glu_single[dmd_key]['image_identity'].keys())

    # determine n_synapses from first image
    first_arr = np.asarray(glu_single[dmd_key]['image_identity'][im_names[0]])
    if first_arr.ndim != 3:
        raise ValueError(f"Unexpected shape for {dmd_key}: {first_arr.shape}")

    # infer synapse axis
    # assume either (n_trials, n_synapses, n_time) or (n_synapses, n_trials, n_time)
    if first_arr.shape[0] < first_arr.shape[1]:
        # likely (trials, synapses, time)
        n_syn = first_arr.shape[1]
        layout = "trials_synapses_time"
    else:
        # likely (synapses, trials, time)
        n_syn = first_arr.shape[0]
        layout = "synapses_trials_time"

    for syn in range(n_syn):
        pre_vals = []
        post_vals = []

        for im in im_names:
            arr = np.asarray(glu_single[dmd_key]['image_identity'][im])

            if arr.ndim != 3:
                raise ValueError(f"Unexpected shape for {dmd_key}, {im}: {arr.shape}")

            if layout == "trials_synapses_time":
                syn_traces = arr[:, syn, :]   # (n_trials, n_time)
            else:
                syn_traces = arr[syn, :, :]   # (n_trials, n_time)

            pre_auc = np.trapz(syn_traces[:, pre_slice], axis=1)
            post_auc = np.trapz(syn_traces[:, post_slice], axis=1)

            pre_vals.extend(pre_auc.tolist())
            post_vals.extend(post_auc.tolist())

        rows.append({
            'dmd': dmd,
            'synapse': syn,
            'pre_auc_mean': np.nanmean(pre_vals),
            'post_auc_mean': np.nanmean(post_vals),
            'delta_auc_mean': np.nanmean(post_vals) - np.nanmean(pre_vals),
            'n_trials_total': len(pre_vals),
        })

pop_df = pd.DataFrame(rows)
pop_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

palette = {1: '#7aa6c2', 2: '#d98c8c'}

for dmd in sorted(pop_df['dmd'].unique()):
    sub = pop_df[pop_df['dmd'] == dmd]
    ax.scatter(
        sub['pre_auc_mean'],
        sub['post_auc_mean'],
        s=10,
        alpha=0.75,
        color=palette[dmd],
        label=f'DMD{dmd}',
        edgecolor='none'
    )

# unity line
xymin = np.nanmin([pop_df['pre_auc_mean'].min(), pop_df['post_auc_mean'].min()])
xymax = np.nanmax([pop_df['pre_auc_mean'].max(), pop_df['post_auc_mean'].max()])
ax.plot([xymin, xymax], [xymin, xymax], color='k', ls='--', lw=1.5)

# ax.set_xlim(xymin, xymax)
# ax.set_ylim(xymin, xymax)

ax.set_xlabel('Mean pre-stimulus AUC')
ax.set_ylabel('Mean post-stimulus AUC')
ax.set_title('Population synapse responses')
ax.legend(frameon=False,handletextpad=0.1,fontsize=12)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

fig.tight_layout()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pre_slice = slice(0, 50)
post_slice = slice(50, 100)

# pooled across both DMDs: one point = one synapse x one image
rows = []

for dmd in [1, 2]:
    dmd_key = f'DMD{dmd}'
    if dmd_key not in glu_single:
        continue

    im_names_dmd = list(glu_single[dmd_key]['image_identity'].keys())

    first_arr = np.asarray(glu_single[dmd_key]['image_identity'][im_names_dmd[0]])
    if first_arr.ndim != 3:
        raise ValueError(f"Unexpected shape for {dmd_key}: {first_arr.shape}")

    # infer layout
    if first_arr.shape[0] < first_arr.shape[1]:
        layout = "trials_synapses_time"
        n_syn = first_arr.shape[1]
    else:
        layout = "synapses_trials_time"
        n_syn = first_arr.shape[0]

    for im in im_names_dmd:
        arr = np.asarray(glu_single[dmd_key]['image_identity'][im])

        if arr.ndim != 3:
            raise ValueError(f"Unexpected shape for {dmd_key}, {im}: {arr.shape}")

        for syn in range(n_syn):
            if layout == "trials_synapses_time":
                syn_traces = arr[:, syn, :]   # (n_trials, n_time)
            else:
                syn_traces = arr[syn, :, :]   # (n_trials, n_time)

            pre_auc = np.trapz(syn_traces[:, pre_slice], axis=1)
            post_auc = np.trapz(syn_traces[:, post_slice], axis=1)

            rows.append({
                'synapse_uid': f'DMD{dmd}_syn{syn}',
                'image': im,
                'pre_auc_mean': np.nanmean(pre_auc),
                'post_auc_mean': np.nanmean(post_auc),
                'delta_auc_mean': np.nanmean(post_auc) - np.nanmean(pre_auc),
                'n_trials': len(pre_auc),
            })

image_pop_df = pd.DataFrame(rows)

# keep image colors consistent
unique_images = image_pop_df['image'].dropna().unique().tolist()
image_palette = {
    im: colors[i % len(colors)]
    for i, im in enumerate(unique_images)
}

fig, ax = plt.subplots(figsize=(4, 4))

sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for im in unique_images:
    sub_im = image_pop_df[image_pop_df['image'] == im]
    ax.scatter(
        sub_im['pre_auc_mean'],
        sub_im['post_auc_mean'],
        s=10,
        alpha=0.75,
        color=image_palette[im],
        label=im.split('/')[-1].split('\\')[-1],
        edgecolor='none'
    )

    # optional: overlay image-wise population mean
#     ax.scatter(
#         np.nanmean(sub_im['pre_auc_mean']),
#         np.nanmean(sub_im['post_auc_mean']),
#         s=140,
#         color=image_palette[im],
#         edgecolor='k',
#         linewidth=1.0,
#         zorder=5
#     )

# unity line
xymin = np.nanmin([image_pop_df['pre_auc_mean'].min(), image_pop_df['post_auc_mean'].min()])
xymax = np.nanmax([image_pop_df['pre_auc_mean'].max(), image_pop_df['post_auc_mean'].max()])
ax.plot([xymin, xymax], [xymin, xymax], color='k', ls='--', lw=1.5)

ax.set_xlim(0, 750)
ax.set_ylim(0, 750)

ax.set_xlabel('Mean pre-stimulus AUC')
ax.set_ylabel('Mean post-stimulus AUC')
ax.set_title('Mean image response')

# handles, labels = ax.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# ax.legend(
#     by_label.values(),
#     by_label.keys(),
#     frameon=False,
#     bbox_to_anchor=(1.02, 1),
#     loc='upper left',
#     fontsize=9
# )

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
fig.tight_layout()
filen = '2026-03-30_population_trial_auc_scatter'
save_figure(fig, os.path.join(savepath, filen), formats=['.png', '.pdf'], dpi=300)

In [ ]:
activation_table_path = glob.glob(os.path.join(test_asset.session_dir,'**','activation_summary_table.csv'),recursive=True)[0]
act_df = pd.read_csv(activation_table_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# load + filter
# -----------------------------

img_df = act_df.loc[act_df['stimulus_family'].str.lower() == 'image'].copy()

# -----------------------------
# collapse to one row per unique synapse_id
# agnostic to DMD
# -----------------------------
def first_or_mode(x):
    x = x.dropna().astype(str)
    if len(x) == 0:
        return np.nan
    mode = x.mode()
    return mode.iloc[0] if len(mode) else x.iloc[0]

syn_df = (
    img_df
    .groupby('synapse_id', as_index=False)
    .agg(
        mean_response_auc=('mean_delta_auc', 'mean'),
        median_response_auc=('median_delta_auc', 'mean'),
        response_class=('response_class', first_or_mode),
        n_rows=('mean_delta_auc', 'size'),
    )
)

# choose which response metric to display
xcol = 'mean_response_auc'

# clean labels a bit in case capitalization varies
syn_df['response_class'] = syn_df['response_class'].astype(str).str.strip().str.lower()

# keep only expected classes
class_order = ['deactivated', 'no_change', 'activated']
syn_df = syn_df[syn_df['response_class'].isin(class_order)].copy()

# -----------------------------
# empirical CDF
# -----------------------------
x = np.sort(syn_df[xcol].to_numpy())
y = np.arange(1, len(x) + 1) / len(x)

# -----------------------------
# class boundaries from data
# -----------------------------
class_bounds = {}
for cls in class_order:
    sub = syn_df.loc[syn_df['response_class'] == cls, xcol]
    if len(sub) > 0:
        class_bounds[cls] = (sub.min(), sub.max())

# overall plotting limits
xmin = syn_df[xcol].min()
xmax = syn_df[xcol].max()
xpad = 0.05 * (xmax - xmin) if xmax > xmin else 1.0

# colors
span_colors = {
    'deactivated': '#4C72B0',  # blue
    'no_change':   'lightgray',  # gray
    'activated':   '#55A868',  # green
}

# -----------------------------
# plot
# -----------------------------
fig, ax = plt.subplots(figsize=(4, 4))
sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

# background spans
for cls in class_order:
    if cls in class_bounds:
        lo, hi = class_bounds[cls]
        ax.axvline(lo, color='white', lw=2, alpha=0.9, zorder=2)
#         ax.axvline(hi, color='white', lw=2, alpha=0.9, zorder=2)
        ax.axvspan(lo, hi, color=span_colors[cls], alpha=0.95, zorder=0)

# cdf line
ax.plot(x, y, color='0.25', lw=2.5, zorder=3)

# optional vertical line at 0


# annotate class labels
# for cls in class_order:
#     if cls in class_bounds:
#         lo, hi = class_bounds[cls]
#         xc = (lo + hi) / 2
#         label = cls.title() if cls != 'no change' else 'No change'
#         ax.text(
#             xc, 1.02, label,
#             ha='center', va='bottom',
#             rotation=30,
#             fontsize=12,
#             weight='bold',
#             color=span_colors[cls],
#             clip_on=False
#         )

# styling
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(-0.02, 1.04)
ax.set_xlabel('Mean response A.U.C')
ax.set_ylabel('CDF')


for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2)


fig.tight_layout()

filen = '2026-03-30_population_cdf'
save_figure(fig, os.path.join(savepath, filen), formats=['.png', '.pdf'], dpi=300)

### Plot mean image response

In [ ]:
mean_data_path = glob.glob(os.path.join(test_asset.derived_dir,'**','glutamate_mean_df.npz'))[0]
glu_qc_path = glob.glob(os.path.join(test_asset.qc_dir,'**','**glutamate**'))[0]

In [ ]:
glu_mean = np.load(mean_data_path,allow_pickle=True)['data'][0]

In [ ]:
with open(glu_qc_path, "r") as f:
    glu_qc = json.load(f)

In [ ]:
im_names = list(glu_mean['DMD1']['image_identity'].keys())
colors = ['#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
 '#e1bee7', '#d7ccc8', '#cfd8dc', '#b2ebf2']

In [ ]:
dmd = 1
syn = 2

fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for i,im in enumerate(im_names):

    glu_signal = glu_mean[f'DMD{dmd}']['image_identity'][im_names[i]]['mean'][syn]
    time = np.linspace(-0.25,0.5,len(glu_signal))
    ax.plot(time,pd.DataFrame(glu_signal).rolling(1,min_periods=1).mean(),lw=3,color=colors[i])

mean = np.mean([glu_mean[f'DMD{dmd}']['image_identity'][im_names[i]]['mean'][syn]for i,im in enumerate(im_names)],axis=0 )

# ax.fill_between(time[50:100],min(mean[50:100]),mean[50:100],color='lightgray',zorder=0,alpha=0.4,label='AUC')
# ax.fill_between(time[:50],min(mean[:50]),mean[:50],color='lightgray',zorder=0,alpha=0.4)

ax.plot(time,pd.DataFrame(mean).rolling(1,min_periods=1).mean(),color='k',lw=2,label='Mean')

ax.axvline(0.0,color='k',dashes=[5,3],lw=2)
ax.axvspan(0.0,0.25,color='lightgray',alpha=0.4,zorder=0)

ax.set_xlabel('Time (s) from stimulus onset')
ax.set_ylabel('Mean \u0394F')

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

ax.set_title('Mean Image Response')
ax.legend(frameon=False,fontsize=12)
fig.tight_layout()

filen = '2026-03-30_AUC1'
# save_figure(fig,os.path.join(savepath,filen),formats=['.png','.pdf'],dpi=300)

### Plot photodiode signal

In [ ]:
pd_path = test_asset.photodiode_pkl
bon_path = test_asset.bonsai_event_log_csv

In [ ]:
stim_df = pd.read_csv(bon_path)
pd_df = pd.read_pickle(pd_path)

In [ ]:
pd_df['time'] = pd_df.index-pd_df.index[0]

In [ ]:
stim_df

In [ ]:
fig,ax=plt.subplots()

ax.tick_params(axis='x',which='major',reset=True,top=False,labelsize=12)
ax.tick_params(axis='y',which='major',reset=True,right=False,labelsize=12)

ax.plot(pd_df['time'],pd_df['AnalogInput0'])

ax2 = ax.twinx()
ax2.tick_params(axis='x',which='major',reset=True,top=False,labelsize=12)
ax2.tick_params(axis='y',which='major',reset=False,right=True,labelsize=12)

ax2.plot(stim_df['corrected_timestamps'].iloc[5:],stim_df['photodiode_state'].iloc[5:],color='red',dashes=[4,3])
ax2.set_ylabel('Photodiode State (BV)')


ax.set_xlim(-0.25,10)
ax2.set_xlim(-0.25,10)
ax.set_xlabel('HARP time (s)')
ax.set_ylabel('Photodiode State (HARP)')

flash_dur_s = 0.25


# use corrected timestamps if present, otherwise raw Bonsai timestamps
time_col = 'corrected_timestamps' if 'corrected_timestamps' in stim_df.columns else 'Timestamp'

# rows corresponding to image presentations
image_mask = (
    stim_df['Value']
    .astype(str)
    .str.endswith('.tiff', na=False)
)

colors = ['#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
 '#e1bee7', '#d7ccc8', '#cfd8dc', '#b2ebf2']

image_mask = stim_df['Value'].astype(str).str.endswith('.tiff', na=False)
image_events = stim_df.loc[image_mask, [time_col, 'Value']].copy()
image_events = image_events.rename(columns={'Value': 'image_name'})
image_events = image_events.sort_values(time_col)

# unique image identities
unique_images = image_events['image_name'].dropna().unique()
# optional: only show spans that overlap current x-limits
image_to_color = {img: colors[i] for i, img in enumerate(unique_images)}

x0, x1 = ax.get_xlim()
for _, row in image_events.iterrows():
    t = row[time_col]
    img = row['image_name']
    if (t + flash_dur_s >= x0) and (t <= x1):
        ax.axvspan(
            t,
            t + flash_dur_s,
            color=image_to_color[img],
            alpha=0.4,
            zorder=0,
        )


ax.set_title('HARP/BV-encoded photodiode signal')
fig.tight_layout()
filen = '2026-03-30_HARP_photodiode_BV'
save_figure(fig,os.path.join(savepath,filen),formats=['.png','.pdf'],dpi=300)

In [ ]:
fig,ax=plt.subplots()

ax.tick_params(axis='x',which='major',reset=True,top=False,labelsize=12)
ax.tick_params(axis='y',which='major',reset=True,right=False,labelsize=12)

ax.plot(stim_df['Timestamp'],stim_df['photodiode_state'])

ax.set_xlim(-0.25,10)
ax.set_xlabel('BonVision time (s)')
ax.set_ylabel('Photodiode State')
ax.set_ylim(-0.1,1.1)
fig.tight_layout()
filen = '2026-03-30_bv_photodiode'
save_figure(fig,os.path.join(savepath,filen),formats=['.png','.pdf'],dpi=300)

### Synapse QC analysis

In [ ]:
qc = []

for asset in assets:
    qc_path = glob.glob(os.path.join(asset.qc_dir,'**','synapse_qc.csv'),recursive=True)[0]
    qc.append(pd.read_csv(qc_path))
qc_df = pd.concat(qc)
qc_df

In [ ]:
qc_df = pd.concat(qc)
qc_df

In [ ]:
# ----------------------------
# helper: concatenate one synapse across trials
# ----------------------------
def get_concatenated_synapse_trace(
    gs,
    dmd,
    synapse_idx,
    trial_list=None,
    trace_type="dF",
    channel=0,
):
    if trial_list is None:
        if hasattr(gs, "valid_trials"):
            trial_list = list(gs.valid_trials[dmd - 1])
        else:
            raise ValueError("Please provide trial_list explicitly.")

    traces = []
    trial_lengths = []
    used_trials = []

    for trial in trial_list:
        arr = gs.get_traces(
            dmd=dmd,
            trial=trial,
            signal=trace_type,
            mode='ls'
        )
        arr = np.asarray(arr)

        if arr.ndim == 3:
            tr = arr[synapse_idx, channel, :]
        elif arr.ndim == 2:
            tr = arr[synapse_idx, :]
        else:
            raise ValueError(f"Unexpected trace shape for trial {trial}: {arr.shape}")

        tr = np.asarray(tr, dtype=float).ravel()
        traces.append(tr)
        trial_lengths.append(len(tr))
        used_trials.append(trial)

    concat_trace = np.concatenate(traces, axis=0)
    return concat_trace, trial_lengths, used_trials

from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.pyplot as plt
import numpy as np

pastel_gyr = LinearSegmentedColormap.from_list(
    "pastel_gyr",
    [
        "#9fd8a3",  # pastel green
        "#f3e7a1",  # pastel yellow
        "#e8a0a0",  # pastel red
    ]
)
# ----------------------------
# choose one session: first session in qc_df
# ----------------------------
# adjust these column names if needed after inspecting qc_df.columns
session_cols_priority = [
    ["session_name"],
    ["session_id"],
    ["mouse_id", "session_date"],
    ["mouse", "session"],
]

session_cols = None
for cols in session_cols_priority:
    if all(c in qc_df.columns for c in cols):
        session_cols = cols
        break

if session_cols is None:
    raise ValueError(f"Could not identify session columns from qc_df.columns:\n{qc_df.columns.tolist()}")

first_session_key = qc_df[session_cols].drop_duplicates().iloc[0].to_dict()

mask = np.ones(len(qc_df), dtype=bool)
for c, v in first_session_key.items():
    mask &= (qc_df[c] == v)

sess_qc = qc_df.loc[mask].copy()

# require DMD + synapse index + quality score
required_cols = ["dmd", "synapse_idx", "quality_score"]
missing = [c for c in required_cols if c not in sess_qc.columns]
if missing:
    raise ValueError(f"Missing required QC columns: {missing}")

sess_qc = sess_qc.dropna(subset=["quality_score"]).sort_values("quality_score").reset_index(drop=True)

if len(sess_qc) < 6:
    raise ValueError(f"Need at least 6 synapses in the selected session, found {len(sess_qc)}")

# ----------------------------
# pick 2 worst, 2 middle, 2 best
# ----------------------------
worst_df = sess_qc.iloc[:2].copy()

mid_start = max(0, len(sess_qc) // 2 - 1)
middle_df = sess_qc.iloc[mid_start:mid_start + 2].copy()

best_df = sess_qc.iloc[-2:].copy()

selected = pd.concat([
    worst_df.assign(group="worst"),
    middle_df.assign(group="middle"),
    best_df.assign(group="best"),
], ignore_index=True)

# optional: order top-to-bottom as best, middle, worst in the plot
selected["group_order"] = selected["group"].map({"best": 0, "middle": 1, "worst": 2})
selected = selected.sort_values(["group_order", "quality_score"], ascending=[True, False]).reset_index(drop=True)

display(selected[["group", "dmd", "synapse_idx", "quality_score"] + [c for c in session_cols if c in selected.columns]])

# ----------------------------
# match to asset
# ----------------------------
# you may need to customize this if your asset object uses different attributes
def asset_matches_session(asset, session_key):
    for c, v in session_key.items():
        if hasattr(asset, c):
            if getattr(asset, c) != v:
                return False
        elif c == "session_name" and hasattr(asset, "session_dir"):
            if os.path.basename(str(asset.session_dir)) != str(v):
                return False
        else:
            # ignore unmatched fields for now
            pass
    return True

candidate_assets = [a for a in assets if asset_matches_session(a, first_session_key)]
if len(candidate_assets) == 0:
    raise ValueError(f"Could not match selected session to an asset using key: {first_session_key}")
asset = candidate_assets[0]

print("Using asset:", getattr(asset, "session_dir", asset))

# ----------------------------
# load glutamate summary
# ----------------------------
# adapt this import / constructor to your current repo
# examples:
# from vip_slap2_analysis.glutamate.summary import GlutamateSummary
# gs = GlutamateSummary(asset.summary_mat_path)

from vip_slap2_analysis.glutamate.summary import GlutamateSummary

# try a few likely asset attrs for the summary path
summary_path_candidates = [
    getattr(asset, "summary_mat_path", None),
    getattr(asset, "glutamate_summary_path", None),
    getattr(asset, "summary_path", None),
    getattr(asset,'summary_mat',None)
]
summary_path_candidates = [p for p in summary_path_candidates if p is not None]

if len(summary_path_candidates) == 0:
    raise ValueError("Could not find summary path on asset. Add your summary path explicitly.")

gs = GlutamateSummary(summary_path_candidates[0])

# ----------------------------
# plot selected synapses
# ----------------------------
fs = 20.0
trace_type = "dF"
channel = 0
offset_step = 150  # vertical separation
plot_seconds = None  # set to e.g. 300 to show only first 5 min

scores = selected["quality_score"].to_numpy()

norm = Normalize(vmin=np.nanmin(scores), vmax=np.nanmax(scores))

# invert so high score -> green, low score -> red
colors = [pastel_gyr(1 - norm(s)) for s in scores]

fig, ax = plt.subplots(figsize=(6, 8))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
n = len(selected)
for i, (_, row) in enumerate(selected.iterrows()):
    dmd = int(row["dmd"])
    synapse_idx = int(row["synapse_idx"])
    q = float(row["quality_score"])
    group = row["group"]

    trace, trial_lengths, used_trials = get_concatenated_synapse_trace(
        gs,
        dmd=dmd,
        synapse_idx=synapse_idx,
        trace_type=trace_type,
        channel=channel,
    )

    t = np.arange(trace.size) / fs
    trace_centered = trace - np.nanmedian(trace)
    y_offset = (n - 1 - i) * offset_step

    ax.plot(
        t,
        trace_centered + y_offset,
        lw=2,
        color=colors[i],
    )

    ax.text(
        t[-1] + 2,
        y_offset,
        f"DMD{dmd} syn {synapse_idx} | score={q:.2f}",
        va="center",
        fontsize=10,
        color=colors[i],
    )

ax.set_xlabel("Time (s)")
ax.set_ylabel("LoCo-derived synapses")
ax.set_title(f"Representative synapse traces\nfrom {asset.session_id}")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(-0.05,5)
ax.set_yticks([])
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
fig.tight_layout()
filen = '2026-03-30_synapse_traces_qc'
save_figure(fig,os.path.join(savepath,filen),formats=['.png','.pdf'],dpi=300)

In [ ]:
qc_df.keys()

In [ ]:
fig,ax=plt.subplots(figsize=(6,4))
sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

p = qc_df[qc_df['recommended_for_analysis']==True]['residual_snr_db']
f = qc_df[qc_df['recommended_for_analysis']==False]['residual_snr_db']

sns.kdeplot(p,label='Pass',lw=2,fill=True,color='b')
sns.kdeplot(f,label='Fail',lw=2,fill=True,color='r')

ax.set_xlabel('Residual SNR (log)')
ax.legend(frameon=False,fontsize=12)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
ax.set_xlim(-12,0)
ax.set_title('QC metric: Residual SNR')
fig.tight_layout()
filen = '2026-03-30_residual_SNR'
save_figure(fig,os.path.join(savepath,filen),formats=['.png','.pdf'],dpi=300)

In [ ]:
fig,ax=plt.subplots(figsize=(6,4))
sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

p = qc_df[qc_df['recommended_for_analysis']==True]['quality_score']
f = qc_df[qc_df['recommended_for_analysis']==False]['quality_score']

sns.kdeplot(p,label='Pass',lw=2,fill=True,color='b')
sns.kdeplot(f,label='Fail',lw=2,fill=True,color='r')

ax.set_xlabel('Sample value range')
ax.legend(frameon=False,fontsize=12)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
# ax.set_xlim(-2000,4000)
fig.tight_layout()